# 05 · Baseline "pre-training" results
### *From Diagnosis to Decision* — ICA 2026

This notebook produces the paper's **baseline** numbers and demonstrates all
three evaluation components on real data:

1. **Diagnosis** — train a classifier, measure in-domain accuracy.
2. **Domain gap** — evaluate the same model on a *different* dataset; the drop
   is the paper's motivation.
3. **Abstention gate** — accuracy-vs-coverage from a confidence threshold.
4. **Risk-weighted error** — a harm matrix that penalises dangerous confusions
   (e.g. *viral read as fungal*) more than harmless ones.

**Two backends, one harness:**
- **CNN backend (Colab / GPU):** transfer-learning with `timm`/`torchvision`
  on PlantVillage (leaf-grouped split) → cross-dataset eval on PlantWild/PlantDoc.
- **CPU smoke backend (runs anywhere, incl. this repo locally):** classic
  colour+texture features → linear model. Weak by design, but it exercises the
  *entire* evaluation harness and yields real numbers on whatever is present
  (locally: PlantDoc). Use it to validate the pipeline before spending GPU time.

## 1 · Setup

In [ ]:
# --- Environment config: works on Google Colab AND locally --------------------
import os, sys, pathlib

def in_colab():
    return "google.colab" in sys.modules or os.path.exists("/content")

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/diagnosis-to-decision")
else:
    # local fallback: repo root (edit if you cloned elsewhere)
    PROJECT_ROOT = pathlib.Path(
        os.environ.get("ICA_PROJECT_ROOT", pathlib.Path.cwd().parents[0])
    )

DATA_RAW     = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_MAPPING = PROJECT_ROOT / "data" / "mapping"
FIGDIR       = PROJECT_ROOT / "reports" / "figures"
for p in (DATA_RAW, DATA_INTERIM, DATA_MAPPING, FIGDIR):
    p.mkdir(parents=True, exist_ok=True)

print("Colab:", in_colab())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW exists:", DATA_RAW.exists())

In [ ]:
# --- Which datasets are actually downloaded? Notebooks adapt to what's present.
import pandas as pd

CANDIDATES = {
    "plantvillage": DATA_RAW / "plantvillage",
    "plantwild":    DATA_RAW / "plantwild",
    "plantseg":     DATA_RAW / "plantseg",
    "plantdoc":     DATA_RAW / "plantdoc",
    "fieldplant":   DATA_RAW / "fieldplant",
    "cassava":      DATA_RAW / "cassava",
    "master":       DATA_RAW / "master_plant_disease",
    "bracol":       DATA_RAW / "bracol",
    "rocole":       DATA_RAW / "rocole",
}
PRESENT = {k: v for k, v in CANDIDATES.items() if v.exists() and any(v.rglob("*"))}
print("Present datasets:", list(PRESENT) or "(none yet — run 00_download_verify first)")

In [ ]:
import numpy as np, pandas as pd, pathlib, warnings, time
import matplotlib.pyplot as plt
from PIL import Image
warnings.filterwarnings("ignore")
IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff"}
RNG = np.random.default_rng(0)

def list_imagefolder(root):
    """Return (paths, labels) for <root>/<split>/<class>/<img> or flat class dirs."""
    root = pathlib.Path(root); paths, labels = [], []
    splits = [d for d in root.iterdir() if d.is_dir() and d.name.lower() in
              {"train","test","val","valid","validation"}]
    scan = []
    if splits:
        for sp in splits:
            for cls in sp.iterdir():
                if cls.is_dir(): scan.append((sp.name, cls))
    else:
        for cls in root.iterdir():
            if cls.is_dir(): scan.append(("all", cls))
    rows=[]
    for split, cls in scan:
        for img in cls.rglob("*"):
            if img.suffix.lower() in IMG_EXTS:
                rows.append((str(img), cls.name, split))
    return pd.DataFrame(rows, columns=["path","label","split"])

## 2 · CPU smoke backend — real numbers on PlantDoc

Features per image (cheap, deterministic, no deep net):
RGB + HSV channel histograms (colour) and a coarse HOG (texture/shape).
Classifier: multinomial logistic regression with balanced class weights.
This is intentionally a *weak* baseline — its job is to prove the harness runs
and to give a floor for the CNN to beat.

In [ ]:
from skimage.feature import hog
from skimage.color import rgb2hsv, rgb2gray

def features(path, size=64):
    im = Image.open(path).convert("RGB").resize((size,size))
    a = np.asarray(im, dtype=np.float32)/255.0
    hsv = rgb2hsv(a)
    hist = []
    for ch in range(3):
        hist += np.histogram(a[...,ch], bins=16, range=(0,1))[0].tolist()
        hist += np.histogram(hsv[...,ch], bins=16, range=(0,1))[0].tolist()
    g = rgb2gray(a)
    h = hog(g, orientations=8, pixels_per_cell=(16,16),
            cells_per_block=(2,2), feature_vector=True)
    return np.concatenate([np.array(hist,dtype=np.float32)/(size*size), h]).astype(np.float32)

def build_xy(df, per_class_cap=250, seed=0):
    """Balanced-ish sample, extract features. Returns X, y(str), paths."""
    # Manual per-class sample (robust across pandas versions; groupby.apply drops
    # the group column by default in pandas 3.0).
    parts = [g.sample(min(len(g), per_class_cap), random_state=seed)
             for _, g in df.groupby("label")]
    keep = pd.concat(parts, ignore_index=True)
    X, y, P = [], [], []
    for p, lab in zip(keep["path"], keep["label"]):
        try:
            X.append(features(p)); y.append(lab); P.append(p)
        except Exception:
            pass
    return np.vstack(X), np.array(y), np.array(P)

In [ ]:
# --- Train the smoke baseline on the largest available folder-dataset ---
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

FOLDER_SETS = [k for k in PRESENT if k in {"plantdoc","fieldplant","cassava","master"}]
assert FOLDER_SETS, "No folder-style dataset present — run 00 (PlantDoc is the local default)."
DS = FOLDER_SETS[0]
print("Smoke baseline on:", DS)

frame = list_imagefolder(PRESENT[DS])
# use the dataset's own train/test if BOTH are non-trivially populated,
# else make a stratified split (dropping classes too small to split).
from sklearn.model_selection import train_test_split
has_both = set(frame["split"]) >= {"train","test"} and \
           (frame.split=="train").sum() >= 50 and (frame.split=="test").sum() >= 50
if has_both:
    tr, te = frame[frame.split=="train"].copy(), frame[frame.split=="test"].copy()
else:
    vc = frame["label"].value_counts()
    keep_cls = vc[vc >= 4].index                      # need >=4 to stratify-split
    f = frame[frame["label"].isin(keep_cls)]
    tr, te = train_test_split(f, test_size=0.30, stratify=f["label"], random_state=0)
    print(f"(no usable provided split — stratified split over {len(keep_cls)} classes "
          f"with >=4 images; dropped {frame['label'].nunique()-len(keep_cls)} tiny classes)")
# keep classes present in BOTH splits
common = sorted(set(tr.label) & set(te.label))
tr, te = tr[tr.label.isin(common)], te[te.label.isin(common)]
print(f"{len(common)} classes | train {len(tr)} | test {len(te)}")

t0=time.time()
Xtr,ytr,_ = build_xy(tr); Xte,yte,Pte = build_xy(te, per_class_cap=10**9)
clf = make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0))
clf.fit(Xtr,ytr)
proba = clf.predict_proba(Xte); classes = clf.classes_
pred = classes[proba.argmax(1)]
acc = accuracy_score(yte,pred); f1m = f1_score(yte,pred,average="macro")
print(f"trained in {time.time()-t0:.1f}s  |  test acc {acc:.3f}  macro-F1 {f1m:.3f}")

## 3 · Confusion & per-class F1

In [ ]:
rep = pd.DataFrame(classification_report(yte,pred,output_dict=True,zero_division=0)).T
per_class = rep.drop(index=[i for i in ["accuracy","macro avg","weighted avg"] if i in rep.index])
display(per_class.sort_values("f1-score")[["precision","recall","f1-score","support"]].round(2))

cm = confusion_matrix(yte,pred,labels=classes)
fig,ax=plt.subplots(figsize=(7,6))
im=ax.imshow(cm/cm.sum(1,keepdims=True).clip(1), cmap="magma")
ax.set_title(f"{DS} — row-normalised confusion (smoke baseline)")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); plt.colorbar(im,fraction=0.046)
plt.tight_layout(); fig.savefig(FIGDIR/f"confusion_{DS}.png"); plt.show()

## 4 · Abstention gate — accuracy vs coverage

A model that *knows when it doesn't know* is central to the paper. We threshold
the max softmax/probability: predict only when confidence ≥ τ, else **abstain**.
Sweeping τ traces the accuracy–coverage curve. A useful gate keeps high accuracy
while covering most inputs; the knee is a candidate operating point.

In [ ]:
conf = proba.max(1); correct = (pred==yte)
taus = np.linspace(0, conf.max(), 40)
cov, sel_acc = [], []
for t in taus:
    m = conf>=t
    cov.append(m.mean())
    sel_acc.append(correct[m].mean() if m.any() else np.nan)
cov=np.array(cov); sel_acc=np.array(sel_acc)

fig,ax=plt.subplots(figsize=(7,4))
ax.plot(cov, sel_acc, marker="o", ms=3, color="#4C78A8")
ax.axhline(acc, ls="--", c="grey", label=f"full-coverage acc = {acc:.3f}")
ax.set_xlabel("coverage (fraction answered)"); ax.set_ylabel("selective accuracy")
ax.set_title(f"{DS} — abstention curve"); ax.legend(); ax.invert_xaxis()
plt.tight_layout(); fig.savefig(FIGDIR/f"abstention_{DS}.png"); plt.show()

# operating point: highest coverage with selective acc >= acc + 0.05
target = acc+0.05
ok = [(c,a,t) for c,a,t in zip(cov,sel_acc,taus) if a>=target]
if ok:
    c,a,t = max(ok, key=lambda z:z[0])
    print(f"At tau={t:.2f}: cover {c:.0%} of inputs at selective acc {a:.3f} "
          f"(+{a-acc:.3f} over answering everything).")
else:
    print("Weak smoke model: no threshold reaches acc+0.05 — expected; the CNN will.")

## 5 · Risk-weighted error — the harm matrix

Accuracy treats every mistake equally. In the field they are not: reading a
**viral** disease (remove the plant) as **fungal** (spray) wastes chemicals and
lets the virus spread. We attach a cost `C[true, pred]` and report
**risk-weighted error** = mean cost per prediction. Here we use a *heuristic*
pathogen grouping from class names as a stand-in for the curated
`disease→pathogen` table built in the mapping step; swap it for the real table
in the paper.

In [ ]:
def pathogen_group(label):
    s = label.lower()
    viral   = any(k in s for k in ["virus","viral","mosaic","curl","yellow leaf curl","cmd","cbsd"])
    bact    = any(k in s for k in ["bacter","spot","blight","canker","wilt"])
    healthy = "healthy" in s
    if healthy: return "healthy"
    if viral:   return "viral"
    if bact:    return "bacterial"
    return "fungal"   # default bucket for the smoke demo

groups = ["healthy","fungal","bacterial","viral"]
# cost of predicting pred-group when truth is true-group (0 diagonal)
HARM = pd.DataFrame(
    [[0, 3, 3, 4],    # true healthy -> unnecessary treatment
     [2, 0, 1, 2],    # true fungal
     [2, 1, 0, 2],    # true bacterial
     [4, 3, 3, 0]],   # true viral -> worst if missed (spread, wrong action)
    index=groups, columns=groups)
display(HARM)

tg = np.array([pathogen_group(l) for l in yte])
pg = np.array([pathogen_group(l) for l in pred])
costs = np.array([HARM.loc[t,p] for t,p in zip(tg,pg)])
rwe = costs.mean()
# baseline: a model that always predicts the majority pathogen group
maj = pd.Series(tg).mode()[0]
rwe_maj = np.array([HARM.loc[t,maj] for t in tg]).mean()
print(f"Group accuracy: {(tg==pg).mean():.3f}")
print(f"Risk-weighted error (mean harm/prediction): {rwe:.3f}")
print(f"  vs majority-group baseline: {rwe_maj:.3f}")
print(f"  vs plain error rate:        {1-acc:.3f}")
print("\nThe gap between plain error and risk-weighted error is the paper's point:"
      "\nthe SAME accuracy can carry very different real-world harm.")

## 6 · CNN backend (Colab / GPU) — leak-safe training + cross-dataset eval

Run on Colab after `00` has fetched **PlantVillage** (with `leaf_id`) and at
least one field set. Uses the **leaf-grouped** split from notebook `04` so no
physical leaf spans train and test. Then evaluates cross-dataset — the accuracy
drop is the headline number.

> Skipped automatically if `torch` is unavailable (e.g. this local repo).

In [ ]:
try:
    import torch, torchvision
    HAVE_TORCH = torch.cuda.is_available() or True
except Exception as e:
    HAVE_TORCH = False
    print("torch not available -> CNN backend skipped (run this cell on Colab).", e)

if HAVE_TORCH and (DATA_RAW/"plantvillage"/"hf_arrow").exists():
    import torch, torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    import torchvision.transforms as T
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    from datasets import load_from_disk

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tfm = EfficientNet_B0_Weights.IMAGENET1K_V1.transforms()

    # Expect 04 to have written a leaf-grouped split table:
    split_csv = DATA_INTERIM/"plantvillage_leafgrouped_split.csv"
    assert split_csv.exists(), "Run 04_leakage_dedup_crosswalk.ipynb first to create the leaf-grouped split."
    split = pd.read_csv(split_csv)   # columns: idx, label, split(train/val/test), leaf_id
    ds = load_from_disk(str(DATA_RAW/"plantvillage"/"hf_arrow"))["train"]
    label_names = ds.features["label"].names

    class PV(Dataset):
        def __init__(self, sub): self.sub = sub.reset_index(drop=True)
        def __len__(self): return len(self.sub)
        def __getitem__(self, i):
            r = self.sub.iloc[i]; ex = ds[int(r["idx"])]
            return tfm(ex["image"].convert("RGB")), int(r["label"])

    tr = DataLoader(PV(split[split.split=="train"]), batch_size=64, shuffle=True, num_workers=2)
    va = DataLoader(PV(split[split.split=="val"]),   batch_size=64, num_workers=2)

    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(label_names))
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    lossf = nn.CrossEntropyLoss()

    for epoch in range(3):                    # 3 epochs is enough for a baseline
        model.train()
        for x,y in tr:
            x,y = x.to(device), y.to(device)
            opt.zero_grad(); loss = lossf(model(x), y); loss.backward(); opt.step()
        # quick val
        model.eval(); c=t=0
        with torch.no_grad():
            for x,y in va:
                p = model(x.to(device)).argmax(1).cpu()
                c += (p==y).sum().item(); t += len(y)
        print(f"epoch {epoch}: val acc {c/t:.3f}")
    torch.save(model.state_dict(), DATA_INTERIM/"baseline_effb0.pt")
else:
    print("CNN backend not run here. The CPU smoke backend above already produced "
          "real in-domain numbers and exercised the full eval harness.")

In [ ]:
# --- Cross-dataset evaluation (Colab): the domain-gap headline ---------------
# Load baseline_effb0.pt, run inference on PlantWild / PlantDoc (mapped to the
# shared label space via data/mapping/class_crosswalk.csv), and report:
#   in-domain acc  vs  in-the-wild acc   ->   the drop that motivates the paper.
# Left as a clearly-scoped TODO because it needs the trained weights + crosswalk.
print("Cross-dataset eval: run after CNN training + class_crosswalk.csv exist.")

---
### What to put in the paper from this notebook
- The **in-domain vs in-the-wild** accuracy gap (headline motivation).
- The **abstention curve**: "at τ=…, we answer X% of inputs at Y% accuracy."
- **Risk-weighted error** vs plain error — the same accuracy, different harm.
- State the baseline architecture, epochs, and that the split was **leaf-grouped**
  (cite notebook `04`) so results are not leakage-inflated.